# Chunk catalogue: one real game, every stream part inspected

The spike toy answered *semantics* (piercing, cached-replay, v2 envelope). This notebook answers
*shapes*: what the chunks actually look like for OUR nodes — the raw material `translate()` will
pattern-match on. One cheap LLM-only game, streamed exactly the way the production server will:

```python
stream_mode=["updates", "custom"], subgraphs=True, version="v2"
```

Three outputs:
1. **Learning**: see every concept from the streaming notes (§1.5 envelope, §1.6 payload shapes, §1.9 subgraph ns) on real data.
2. **Exhaustiveness check**: the observed node→keys inventory, to be read against `frontend/event_derivation.md` — every key must be an event row or an explicit fold.
3. **Fixture**: `notebooks/fixtures/chunk_catalogue.jsonl` — the translator's offline dev data + the frontend's first mock.

Companion reading: the streaming notes, Part 1. Where the notes and this output disagree, **this output wins** — it's our pinned version's real behavior.

## Setup + provenance

Stamp everything a future reader needs to trust this record: langgraph version, git SHA, and the
active model (confirm it's the cheap one BEFORE running the game cell — this notebook costs one game).

In [ ]:
import sys, pathlib

# Kernel cwd is notebooks/; put the repo root on sys.path so `Agents` imports
# (same root cause as needing `python -m pytest` instead of bare `pytest`).
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json, os, subprocess
from importlib.metadata import version
from dotenv import load_dotenv

load_dotenv(ROOT / ".env")
print("repo root:", ROOT)
print("langgraph:", version("langgraph"))
print("git SHA:  ", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
print("model env:", os.environ.get("GOOGLE_GENAI_MODEL", "<unset — check llm_factory default>"))

## Settle the notes' unverified claim first

The streaming notes claim v2 changes `invoke()`'s return type to a `GraphOutput`. Ten-second check:
does our `invoke` even take `version=`? (The notes' other contested row — interrupts moving to
`values` parts under v2 — was already refuted empirically in the spike notebook: on 1.1.10 the
interrupt arrives as `__interrupt__` inside an **updates** part.)

In [ ]:
import inspect
from Agents.graphs.parent import parent_graph_compiled

sig = inspect.signature(parent_graph_compiled.invoke)
print("invoke has version param:", "version" in sig.parameters)
print(sig)

## Build the run exactly like production

Mirrors `run_game` (Agents/main.py): same `RunConfig` normalization, same `build_runnable_config`,
same `context`. Differences: no human seat (LLM-only — the spike already covered interrupts),
`dump_enabled=False` (a spike game must never be mined into the memory store), and no Langfuse
root-span ceremony (the stream is the record here).

In [ ]:
from Agents.config import RunConfig, build_runnable_config, normalize_run_config
from Agents.main import INITIAL_STATE
from Agents.memory import store
from Agents.memory.persistence import seed_memory_from_config
from Agents.observability import EvalCaseSink
from Agents.run_fingerprint import runtime_fingerprint
from Agents.tracing import Metrics

run = normalize_run_config(RunConfig(memory_persistence={"dump_enabled": False}))
seed_memory_from_config(run.memory_persistence, target_store=store)
config = build_runnable_config(run, metadata={"runtime_fingerprint": runtime_fingerprint()})
context = {"metrics": Metrics(), "eval_sink": EvalCaseSink()}
initial_state = {key: value.copy() for key, value in INITIAL_STATE.items()}
print("game_id:", config["configurable"]["game_id"])

## THE game cell — costs one cheap game (~1-2 min)

Every part is captured verbatim AND written incrementally to the fixture file (so a crash
mid-game still leaves a usable partial). We print only a rolling sample live — the full
inspection happens in the analysis cells below, offline.

Serializer note: `data` contains Pydantic objects (WolfChannel, DeathRecord…) — `model_dump()`
where available, `repr` otherwise. `ns` tuples become lists (JSON has no tuples).

In [ ]:
import pathlib

FIXTURE = pathlib.Path("fixtures/chunk_catalogue.jsonl")
FIXTURE.parent.mkdir(exist_ok=True)


def _jsonable(obj):
    if hasattr(obj, "model_dump"):
        return obj.model_dump(mode="json")
    if isinstance(obj, dict):
        return {k: _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    return repr(obj)


parts = []
with FIXTURE.open("w") as f:
    header = {"langgraph": version("langgraph"),
              "git": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
              "stream": {"stream_mode": ["updates", "custom"], "subgraphs": True, "version": "v2"}}
    f.write(json.dumps({"_header": header}) + "\n")
    for part in parent_graph_compiled.stream(
        initial_state, config=config, context=context,
        stream_mode=["updates", "custom"], subgraphs=True, version="v2",
    ):
        parts.append(part)
        f.write(json.dumps({"i": len(parts) - 1, "type": part["type"],
                            "ns": list(part["ns"]), "data": _jsonable(part["data"])}) + "\n")
        if len(parts) <= 8 or len(parts) % 25 == 0:
            print(f"[{len(parts):4d}]", part["type"], part["ns"], str(part["data"])[:110])

print(f"\ncaptured {len(parts)} parts -> {FIXTURE}")

## Analysis 1 — taxonomy: what kinds of parts exist?

Notes §1.5: every part is `{type, ns, data}`. Count by `type`, and for updates, by the node name
(the single key of `data`). **What to look for:** `discuss`/`vote` appearing many times (the
generic day actors), the phase-marker nodes appearing once per day, `custom` parts (that's
`turn_started` from the route_speaker EDGE — a writer emitting from an edge, notes §1.7), and
`__interrupt__` appearing zero times (no human seat).

In [ ]:
from collections import Counter

print(Counter(p["type"] for p in parts))
print()
node_counts = Counter(
    node for p in parts if p["type"] == "updates"
    for node in p["data"] if not node.startswith("__")
)
for node, n in node_counts.most_common():
    print(f"{n:4d}  {node}")

## Analysis 2 — namespaces: where subgraph parts come from

Notes §1.9: `ns == ()` is the root graph; a non-empty tuple is a subgraph, formatted
`"NODE_NAME:task-id"` — stable name before the colon, per-invocation id after. **What to look
for:** the wolf subgraph's internals (`PREPARE_WOLF_NIGHT`, `WOLF_NIGHT_DISCUSS`…) arriving under
`('WOLF_NIGHT_PHASE:…',)` — this is the dispatch key the translator uses to attribute wolf-channel
chunks. The translator must match on the part BEFORE the colon only (the task-id changes every night).

In [ ]:
ns_shapes = Counter(
    tuple(seg.split(":")[0] for seg in p["ns"]) for p in parts
)
for shape, n in ns_shapes.most_common():
    print(f"{n:4d}  {shape or '(root)'}")

print("\nsample subgraph part:")
sub = next((p for p in parts if p["ns"]), None)
print(sub if sub else "none observed — check subgraphs=True was passed")

## Analysis 3 — the key inventory (the exhaustiveness check)

For every node: which state keys did it commit? This table is the translator's contract surface.
**Read it against `frontend/event_derivation.md`:** every (node, key) pair must be either an event
row or an explicitly-ruled fold/IGNORED. Anything unaccounted for is a hole in the doc — found
now, before the translator hardcodes the omission. Record verdicts in the final cell.

In [ ]:
from collections import defaultdict

inventory = defaultdict(set)
for p in parts:
    if p["type"] != "updates":
        continue
    scope = "/".join(seg.split(":")[0] for seg in p["ns"]) or "root"
    for node, delta in p["data"].items():
        if node.startswith("__") or delta is None:
            continue
        inventory[(scope, node)].update(delta.keys() if isinstance(delta, dict) else [f"<non-dict: {type(delta).__name__}>"])

for (scope, node), keys in sorted(inventory.items()):
    print(f"{scope:24s} {node:28s} {sorted(keys)}")

## Analysis 4 — the custom channel: `turn_started`

Notes §1.7: `custom` carries whatever a `get_stream_writer()` writer emitted. Ours fires from the
`route_speaker` conditional EDGE (the double-fire-proof placement, ruled 2026-07-29). **What to
look for:** payload is exactly `(player, day)` — the leak-audit surface for
`check_event_stream_isolation` is that nothing role-gated ever rides this channel.

In [ ]:
customs = [p for p in parts if p["type"] == "custom"]
print(f"{len(customs)} custom parts; first three:")
for p in customs[:3]:
    print(" ", p)

## Verdicts (fill in after reading the inventory against the derivation doc)

- langgraph version / git SHA: `____` / `____`
- `invoke` has `version` param on 1.1.10: YES / NO → notes' GraphOutput row: applies / does not apply to our pin
- Node/key pairs NOT accounted for by the derivation doc (holes): `____`
- Doc rows never observed in this game (fine if role absent this cast — note why): `____`
- Wolf-subgraph ns prefix confirmed as `WOLF_NIGHT_PHASE`: YES / NO
- `turn_started` payload = (player, day) only: YES / NO
- Fixture: `notebooks/fixtures/chunk_catalogue.jsonl`, `____` parts — the translator dev fixture + frontend mock source.

**Caveat for the record:** one game does not exercise every path (vigilante may hold bullets, SK
whiff may not occur, no human seat → no interrupt parts). The inventory proves what WAS seen,
never that nothing else exists — the derivation doc remains the authority on the full surface.